In [ ]:
# -*- coding: utf-8 -*-
# W15-D4 标准字体配置（TOOLS.md 方式）
from matplotlib import font_manager
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import os, re, json, yaml, collections
from pathlib import Path

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)

W15 = "/root/learning-notebooks/第15周"
CONSTRAINTS_YAML = "/root/learning-notebooks/semantic-model/policy/ai-execution-constraints-v0.1-draft.yaml"
LNKCRE = "/root/lnkcre/backend/internal"


# ⚡ W15-D4 · Policy 语义化：审批流 → Policy Model 抽取规则 × AI 执行约束声明验证

> 开发期 · Week 15 Day 4（2026-09-10 周四）
> Today's Question：**AI 需要知道"谁审批"，还是需要知道"为什么找他审批"？**
> 实验设计：不复刻 md 的叙述，用**可执行实验**验证四件事——
> ① mi 的受限谓词与审批链解析器语义可以脱离 Go 独立复现（回放 openspec 8 个官方场景）；
> ② "谁"塌缩"为什么"：同一终审级别下存在多种独立理由路径，只答"谁"必然丢因；
> ③ 三处断链不是叙述而是可复现行为（升级意图丢失→误路由率；门禁不回填→审批人错配率；grep 实证零消费）；
> ④ 今天的主交付物 ai-execution-constraints-v0.1-draft.yaml 是机器可校验的（封闭词表/证据锚/占位符一致性全过）。

**对应当日 md**：`第15周-Day4-Policy语义化与AI执行约束声明格式.md`（阅读材料）；本 notebook 是概念实验版。

## §1 受限谓词求值器复刻（对应 `workflow/condition_evaluator.go` + `approvalmatrix.evalPredicate`）

Go 事实：8 个比较符（`>,>=,<,<=,==,!=,in,not_in`）+ `and`/`or` 二元组合；**字段解析失败返回 False（保守跳过）而非报错**；非法操作符才报 `ErrInvalidConditionExpression`。这正是 ADR-004 §9"可机械评审、非图灵完备"的工程形态——谓词深度由 JSON 结构决定，无循环无递归求值副作用。

In [ ]:
OPS = {">", ">=", "<", "<=", "==", "!=", "in", "not_in"}

class InvalidConditionExpression(Exception):
    pass

def evaluate_condition(cond, doc):
    """复刻 workflow.EvaluateCondition：空条件→True；坏 JSON/坏操作符→异常；字段缺失→False。"""
    if not cond:
        return True
    if isinstance(cond, str):
        try:
            cond = json.loads(cond)
        except json.JSONDecodeError as e:
            raise InvalidConditionExpression(f"malformed JSON: {e}")
    return _pred(cond, doc)

def _pred(raw, doc):
    if "and" in raw:
        return all(_pred(c, doc) for c in raw["and"])
    if "or" in raw:
        return any(_pred(c, doc) for c in raw["or"])
    return _cmp(raw, doc)

def _cmp(raw, doc):
    field, op = raw.get("field"), raw.get("op")
    if field is None or op is None:
        raise InvalidConditionExpression("missing field/op")
    if op not in OPS:
        raise InvalidConditionExpression(f"unsupported op {op!r}")
    if field not in doc:          # Go 语义：DocumentFieldLookup 解析不到 → False，不报错
        return False
    v, val = doc[field], raw["value"]
    if op == ">":  return v > val
    if op == ">=": return v >= val
    if op == "<":  return v < val
    if op == "<=": return v <= val
    if op == "==": return v == val
    if op == "!=": return v != val
    if op == "in": return v in val
    return v not in val           # not_in

# —— 语义回归（对照 Go 行为逐条断言）——
doc = {"amount": 75000, "term_years": 3, "area": 450}
assert evaluate_condition("", doc) is True                                   # 空条件直通
assert evaluate_condition({"field":"amount","op":">","value":50000}, doc) is True
assert evaluate_condition({"field":"amount","op":"<=","value":50000}, doc) is False
assert evaluate_condition({"and":[{"field":"amount","op":">","value":50000},
                                  {"field":"term_years","op":"<","value":5}]}, doc) is True
assert evaluate_condition({"or":[{"field":"area","op":"<","value":100},
                                 {"field":"area","op":">","value":300}]}, doc) is True
assert evaluate_condition({"field":"risk_level","op":"==","value":"high"}, doc) is False  # 字段缺失→False（保守）
assert evaluate_condition({"field":"term_years","op":"in","value":[3,5,7]}, doc) is True
try:
    evaluate_condition({"field":"amount","op":"regex","value":".*"}, doc)
    raise AssertionError("非法操作符未被拒绝")
except InvalidConditionExpression:
    pass
print("§1 受限谓词求值器：8 操作符 + and/or + 字段缺失→False + 非法操作符→异常，全部语义回归通过")

## §2 审批链解析器复刻（对应 `approvalmatrix/service.go resolveChain`）+ openspec 8 场景回放

Go 事实：`project→city→hq` 固定有序遍历；每级四道闸（MinLevel 下限 / 带约束 / auto_route 谓词 / threshold_max），**跳过的级别记入 `EscalatedFrom` 追迹**；`amount < threshold_min` 记 `AutoPassEligible`；全不覆盖时终端兜底 hq（若 hq 行过带+谓词闸则优先带 rule_id——注意 Go 终端兜底**不再复查 threshold_max**，此处忠实镜像）。

In [ ]:
LEVELS = ["project", "city", "hq"]

def rule(rid, tmin, tmax, active=True, area=None, term=None, auto_route=None):
    return {"id": rid, "threshold_min": tmin, "threshold_max": tmax, "active": active,
            "area": area, "term": term, "auto_route": auto_route}

def _bands_cover(r, doc):
    amin, amax = (r["area"] or (None, None))
    if amin is not None and (doc.get("area") is None or doc["area"] < amin): return False
    if amax is not None and (doc.get("area") is None or doc["area"] > amax): return False
    tmin, tmax = (r["term"] or (None, None))
    if tmin is not None and (doc.get("term_years") is None or doc["term_years"] < tmin): return False
    if tmax is not None and (doc.get("term_years") is None or doc["term_years"] > tmax): return False
    return True

def _route_ok(r, doc):
    return r["auto_route"] is None or evaluate_condition(r["auto_route"], doc)

def resolve_chain(rules, doc, min_level=None):
    """rules: {level: rule|None}。返回 level/rule_id/auto_pass/skips（EscalatedFrom 的逐级原因版）。"""
    if not any(rules.values()):
        return {"level": None, "rule_id": None, "auto_pass": False, "skips": ()}   # spec: 无配置→null（200 不是错）
    skips, min_rank = [], (LEVELS.index(min_level) if min_level else -1)
    for i, lv in enumerate(LEVELS):
        if min_rank >= 0 and i < min_rank:
            skips.append(("minlevel_floor", lv)); continue
        r = rules.get(lv)
        if r is None or not r["active"]:
            continue                                    # Go: !ok || !IsActive → 直接 continue，不记迹
        if not _bands_cover(r, doc):
            skips.append(("band_skip", lv)); continue
        if not _route_ok(r, doc):
            skips.append(("predicate_skip", lv)); continue
        if r["threshold_max"] is not None and doc["amount"] > r["threshold_max"]:
            skips.append(("threshold_escalation", lv)); continue
        return {"level": lv, "rule_id": r["id"],
                "auto_pass": doc["amount"] < r["threshold_min"], "skips": tuple(skips)}
    hq = rules.get("hq")                                # Go 终端兜底：只复查带+谓词，不复查 threshold_max
    if hq and hq["active"] and _bands_cover(hq, doc) and _route_ok(hq, doc):
        return {"level": "hq", "rule_id": hq["id"],
                "auto_pass": doc["amount"] < hq["threshold_min"], "skips": tuple(skips)}
    return {"level": "hq", "rule_id": None, "auto_pass": False, "skips": tuple(skips)}

# —— 回放 approval-authority-matrix spec 的 8 个官方场景 ——
base = {"project": rule(101, 0, 50000), "city": rule(201, 0, 200000), "hq": rule(301, 0, None)}
r1 = resolve_chain(base, {"amount": 30000})
assert (r1["level"], r1["auto_pass"], r1["rule_id"]) == ("project", False, 101)          # 场景1
r2 = resolve_chain(base, {"amount": 75000})
assert r2["level"] == "city" and r2["skips"] == (("threshold_escalation", "project"),)   # 场景2
r3 = resolve_chain(base, {"amount": 5_000_000})
assert r3["level"] == "hq"                                                              # 场景3
r4 = resolve_chain({"project": rule(101, 500, 5000), "city": None, "hq": None}, {"amount": 200})
assert r4["level"] == "project" and r4["auto_pass"] is True                             # 场景4 auto_pass
r5 = resolve_chain({"project": None, "city": None, "hq": None}, {"amount": 1000})
assert r5["level"] is None                                                              # 场景5 无配置→null
r6 = resolve_chain({"project": rule(101, 0, 50000), "city": None, "hq": rule(301, 0, None)}, {"amount": 75000})
assert r6["level"] == "hq" and ("threshold_escalation", "project") in r6["skips"]        # 场景6 缺级不堵升级
r7 = resolve_chain({"project": rule(101, 0, 100000, area=(100, 300)), "city": rule(201, 0, 200000), "hq": rule(301, 0, None)},
                   {"amount": 80000, "area": 450})
assert r7["level"] == "city" and ("band_skip", "project") in r7["skips"]                 # 场景7 面积带收窄
r8 = resolve_chain({"project": rule(101, 0, 100000, auto_route={"field": "term_years", "op": ">", "value": 5}),
                    "city": rule(201, 0, 200000), "hq": rule(301, 0, None)}, {"amount": 40000, "term_years": 3})
assert r8["level"] == "city" and ("predicate_skip", "project") in r8["skips"]            # 场景8 谓词跳级
r9 = resolve_chain(base, {"amount": 30000}, min_level="city")                            # SAL-021 红旗下限
assert r9["level"] == "city" and r9["skips"] == (("minlevel_floor", "project"),)
print("§2 审批链解析器：openspec 8 官方场景 + SAL-021 MinLevel 下限，共 9 项断言全部回放通过")

## §3 核心实验："谁"塌缩"为什么"（Today's Question 的量化）

设计一个三闸齐全的 project 级规则（金额上限 20 万 / 面积带 100-300㎡ / 谓词 term_years>5），city 级 0-100 万无附加闸，hq 兜底。扫描金额×面积×租期×红旗 标记的单据网格——**同一个终审级别 city 会由四种互斥的"为什么"路径产生**：金额越带升级、面积带跳级、谓词跳级、红旗下限强制。只答"谁"（city）的 AI 把 4 条政策理由压成 1 个标签。

In [ ]:
matrix = {
    "project": rule(101, 0, 200_000, area=(100, 300), auto_route={"field": "term_years", "op": ">", "value": 5}),
    "city":    rule(201, 0, 1_000_000),
    "hq":      rule(301, 0, None),
}
amounts  = [30_000, 120_000, 199_999, 250_000, 450_000, 900_000, 1_500_000]
areas    = [150, 450]        # 150 在带内；450 越带
terms    = [3, 7]            # 3 谓词 False；7 通过
reds     = [False, True]
docs = [{"amount": a, "area": ar, "term_years": t, "red": rd}
        for a in amounts for ar in areas for t in terms for rd in reds]
results = [(d, resolve_chain(matrix, d, min_level="city" if d["red"] else None)) for d in docs]

by_level = collections.defaultdict(list)
for d, res in results:
    by_level[res["level"]].append((d, res))

rows = []
for lv in ["project", "city", "hq", None]:
    if lv not in by_level: continue
    group = by_level[lv]
    whys = collections.Counter(res["skips"] for _, res in group)
    rows.append((lv or "null", len(group), len(whys), [f"{'/'.join(f'{r}@{l}' for r,l in k)}" if k else "直配无跳级" for k in whys]))

city_whys = {res["skips"] for _, res in by_level["city"]}
assert len(city_whys) >= 4, f"city 级独立理由路径不足 4 条: {len(city_whys)}"
print(f"{'终审级别':<8}{'单据数':>6}{'独立why路径数':>12}")
for lv, n, k, _ in rows:
    print(f"{lv:<8}{n:>6}{k:>12}")
print("\ncity 级的 4 条独立理由路径（同'谁'不同'为什么'）:")
for w in sorted(city_whys, key=str):
    print("  ", " ← ".join(f"{reason}@{level}" for reason, level in w) if w else "  (直配)")
WHO_COLLAPSE = {lv: k for lv, _, k, _ in rows}

## §4 断链三处模拟（md §2 的行为学证据）

- **断链①（升级意图丢失）**：红旗单据本应 min_level=city；若 StartInput.MinApprovalLevel 在引擎侧被丢弃（现状），解析按无下限执行 → 红旗单据落到 project 级的比例即**误路由率**。
- **断链②（门禁不回填）**：workflowpolicy.Resolver 解析出策略角色（示例：role 12），但 SubmitForApproval 丢弃返回值，节点分配仍是 seed 硬编码 role_id=1 → **审批人错配率**。
- **断链③（声明无执行）**：对真实仓库做 grep 实证——MinApprovalLevel 在 workflow 包内除声明行外零消费；AuthorizationPolicy 无执法点。

In [ ]:
# —— 断链①：红旗误路由率 ——
red_results = [(d, resolve_chain(matrix, d, min_level="city"),
                     resolve_chain(matrix, d, min_level=None))      # 模拟引擎丢弃下限
               for d in docs if d["red"]]
misrouted = [(d, dropped) for d, intended, dropped in red_results if dropped["level"] == "project"]
MISROUTE_RATE = len(misrouted) / len(red_results)
print(f"断链① 红旗单据 {len(red_results)} 份：丢下限后落回 project 级 {len(misrouted)} 份，误路由率 {MISROUTE_RATE:.0%}")
for d, res in misrouted[:2]:
    print(f"   例: amount={d['amount']:>8,} area={d['area']} term={d['term_years']} → 解析为 {res['level']}（应为 city）")

# —— 断链②：审批人错配率 ——
policy_role, seed_node_role = 12, 1        # workflowpolicy 映射到 role 12；seed.go 硬编码 role_id=1
assign_docs = [d for d in docs if not d["red"]]     # 非红旗单据会真实到达 project_review 节点
mismatch = sum(1 for _ in assign_docs)              # ResolveRole 返回值被丢弃 → 每一份的节点审批人都是 seed 角色
print(f"断链② 策略映射 role={policy_role} vs seed 节点 role={seed_node_role}："
      f"{len(assign_docs)} 份到达项目审核的单据审批人错配 {mismatch} 份（错配率 100%——门禁只验存在性不回填）")

# —— 断链③：仓库 grep 实证（不是叙述，是文件系统证据）——
def hits(root, needle):
    out = []
    for p in sorted(Path(root).rglob("*.go")):
        if p.name.endswith("_test.go"): continue
        n = p.read_text(errors="ignore").count(needle)
        if n: out.append((str(p.relative_to(Path(root).parent)), n))
    return out

ml_hits = hits(LNKCRE, "MinApprovalLevel")
wf_consumers = [(f, n) for f, n in ml_hits if "/workflow/" in f and not f.endswith("workflow/model.go")]
ap_hits = hits(LNKCRE, "AuthorizationPolicy")
ap_enforcers = [(f, n) for f, n in ap_hits if "/workflow/" in f]
assert not wf_consumers, f"workflow 包内发现意外消费者: {wf_consumers}"
assert not ap_enforcers, f"workflow 包内发现 authorization_policy 执法点: {ap_enforcers}"
print("断链③ grep 实证：MinApprovalLevel 全部出现位置 →")
for f, n in ml_hits: print(f"   {f} ×{n}")
print("   workflow 包内除 model.go 声明行外零消费 ✓")
print("   AuthorizationPolicy 出现位置 →", ", ".join(f for f, _ in ap_hits) or "无", "｜workflow 引擎内执法点 0 ✓")

## §5 主交付物机检：ai-execution-constraints-v0.1-draft.yaml

校验规则（R1-R5 的机器化）：封闭词表（type/ai_may/ai_may_not/fail_mode/谓词符/适用维度）、证据四必填键（carrier/business_key/rationale_fields/version_binding）、**解释模板占位符 ⊆ rationale_placeholders**（R5 prompt 通道的自洽性）、draft 状态与缺口登记完整性。

In [ ]:
spec = yaml.safe_load(open(CONSTRAINTS_YAML))
schema = spec["schema"]; problems = []

def check(cond, msg):
    if not cond: problems.append(msg)

check(spec["meta"]["status"] == "draft-proposal", "meta.status 应为 draft-proposal（未评审不得转正）")
check(len(spec["known_gaps"]) >= 3, "known_gaps 应登记 ≥3 条（GAP-P1/P2/P3）")

for c in spec["constraints"]:
    cid = c["id"]
    check(c["type"] in schema["constraint_types"], f"{cid}: type 不在封闭词表")
    check(set(c["rules"]["ai_may"]) <= set(schema["ai_may"]), f"{cid}: ai_may 越界")
    check(set(c["rules"]["ai_may_not"]) <= set(schema["ai_may_not"]), f"{cid}: ai_may_not 越界")
    check(c["rules"]["fail_mode"] in schema["fail_modes"], f"{cid}: fail_mode 越界")
    check(set(c["applies_to"]) <= set(schema["applicability_dims"]), f"{cid}: 适用维度越界")
    for dim in ("threshold", "custom_predicate"):
        if dim in c["applies_to"]:
            check(c["applies_to"][dim]["op"] in schema["predicate_ops"], f"{cid}: {dim} 谓词符越界（R1）")
    ev = c["evidence"]
    check(set(schema["evidence_required"]) <= set(ev), f"{cid}: 证据缺必填键 {set(schema['evidence_required'])-set(ev)}")
    ph_in_tpl = set(re.findall(r"\{(\w+)\}", c["explanation"]["template"]))
    ph_declared = set(c["explanation"]["rationale_placeholders"])
    check(ph_in_tpl <= ph_declared, f"{cid}: 模板占位符未声明 {ph_in_tpl - ph_declared}")
    check(c["rules"]["fail_mode"] != "open_explicit" or c["type"] == "act_with_approval",
          f"{cid}: fail-open 只允许出现在 act_with_approval 且须显式人定来源")

assert not problems, "约束声明机检失败:\n" + "\n".join(problems)
print(f"§5 约束声明机检：{len(spec['constraints'])} 条约束实例 × 封闭词表/证据四键/占位符一致性/fail-open 限定，全部通过")
print("   词表规模：", {k: len(v) for k, v in schema.items()})

## §6 可视化：升级链全景 & "谁"塌缩"为什么"

In [ ]:
# 图1：金额→级别阶梯 + 红旗下限效应
fig, ax = plt.subplots(figsize=(9, 4.6))
xs = np.logspace(4, np.log10(2_000_000), 400)
lv_idx = {"project": 0, "city": 1, "hq": 2}
normal = [lv_idx[resolve_chain(matrix, {"amount": x, "area": 150, "term_years": 7})["level"]] for x in xs]
red    = [lv_idx[resolve_chain(matrix, {"amount": x, "area": 150, "term_years": 7}, min_level="city")["level"]] for x in xs]
ax.step(xs, normal, where="post", lw=2.2, label="普通单据（term=7, 面积带内）")
ax.step(xs, red,    where="post", lw=2.2, ls="--", color="crimson", label="红旗单据（MinLevel=city 下限）")
for amt in (200_000, 1_000_000):
    ax.axvline(amt, color="gray", ls=":", lw=1)
    ax.text(amt, 2.35, f"{amt/10000:.0f}万", ha="center", fontsize=9, color="dimgray")
ax.set_xscale("log"); ax.set_yticks([0, 1, 2], ["project", "city", "hq"])
ax.set_xlabel("单据金额（元，对数轴）"); ax.set_ylabel("终审级别")
ax.set_title("审批升级链：金额→级别（approvalmatrix resolveChain 复刻）与红旗下限效应")
ax.legend(loc="upper left", fontsize=9); fig.tight_layout()
fig.savefig(f"{W15}/w15d4_升级链与红旗下限.png", dpi=150); plt.close(fig)

# 图2：同一终审级别下的独立理由路径数（信息塌缩）
fig, ax = plt.subplots(figsize=(8, 4.4))
levels = [lv for lv in ["project", "city", "hq"] if lv in WHO_COLLAPSE]
vals = [WHO_COLLAPSE[lv] for lv in levels]
counts = [len(by_level[lv]) for lv in levels]
bars = ax.bar(levels, vals, color=["#4c78a8", "#f58518", "#54a24b"], width=0.55)
for b, v, n in zip(bars, vals, counts):
    ax.text(b.get_x() + b.get_width()/2, v + 0.06, f"{v} 条理由\n({n} 份单据)", ha="center", fontsize=10)
ax.set_ylabel("独立 why 路径数（skips 组合种类）")
ax.set_title("'谁'塌缩'为什么'：同一终审级别下互斥的政策理由路径数")
ax.set_ylim(0, max(vals) + 1.1); fig.tight_layout()
fig.savefig(f"{W15}/w15d4_同级多因塌缩.png", dpi=150); plt.close(fig)
print("图已落盘：w15d4_升级链与红旗下限.png / w15d4_同级多因塌缩.png")

## §7 结论（Today's Question 的数字版答案）

1. **"谁"塌缩"为什么"**：city 级一个标签下面压着 ≥4 条互斥理由路径（金额越带 / 面积带跳级 / 谓词跳级 / 红旗下限）——只答"谁"的 AI 必然丢因。
2. **"谁"随配置漂移，"为什么"锚住证据**：改角色映射，审批人变、级别与理由不变；改阈值，"为什么"能精确指出哪些单据翻转、为什么翻。
3. **系统自己正丢着"为什么"**：红旗误路由率见 §4、审批人错配 100%、两处 grep 零消费——AI 若只学"谁"，会在断链处复读错误答案；学"为什么"才能说"应升级但系统未执行"。约束声明格式把 rationale_fields/escalated_from 列为一等公民，就是为了让"为什么"有唯一可信的取数处。

In [ ]:
summary = {
    "spec场景回放": "8+1 项断言全过",
    "city级独立why路径": len(city_whys),
    "红旗误路由率(断链①)": f"{MISROUTE_RATE:.0%}",
    "审批人错配率(断链②)": "100%（门禁不回填）",
    "断链③grep": "MinApprovalLevel 引擎零消费 / AuthorizationPolicy 零执法",
    "约束声明机检": f"{len(spec['constraints'])} 条全过（封闭词表+证据四键+占位符一致性）",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert len(city_whys) >= 4 and not problems and MISROUTE_RATE > 0
print("\nW15-D4 实验全部断言通过 ✓")